### Criação do schema Gold

Cria o schema da camada Gold, onde ficarão as tabelas prontas para análise e consumo.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS voebem.gold;

### Conferência da tabela Silver

Consulta o schema da tabela de voos na Silver antes de iniciar a construção da fato.

In [0]:
spark.sql("""
    DESCRIBE TABLE voebem.silver.vra
""").show(truncate=False)

### Criação da fato de voos

Cria a tabela base de voos a partir da Silver, removendo registros totalmente duplicados.

In [0]:
fato_voos = spark.sql("""
    SELECT DISTINCT *
    FROM voebem.silver.vra
""")

display(fato_voos.limit(10))

### Validação da quantidade de registros

Compara a quantidade de registros da Silver com a quantidade de registros que será usada na fato.

In [0]:
print("Silver:", spark.table("voebem.silver.vra").count())
print("Fato:", fato_voos.count())

### Gravação inicial da fato

Salva a fato de voos no schema Gold para começar as transformações e validações.

In [0]:
fato_voos.write \
    .mode("overwrite") \
    .saveAsTable("voebem.gold.fato_voos")

### Validação da fato

Confere a quantidade de linhas gravadas na tabela Gold.

In [0]:
display(spark.sql("""
    SELECT COUNT(*) AS total_linhas
    FROM voebem.gold.fato_voos
"""))

### Indicador de partida pontual

Cria um indicador para identificar se a partida ocorreu dentro do limite de 15 minutos de atraso.

In [0]:
from pyspark.sql import functions as F

fato_voos = fato_voos.withColumn(
    "partida_pontual",
    F.when(
        F.col("partida_real").isNull(),
        None
    ).otherwise(
        F.col("atraso_partida_min") <= 15
    )
)

display(
    fato_voos.select(
        "situacao_voo",
        "atraso_partida_min",
        "partida_real",
        "partida_pontual"
    ).limit(20)
)

### Indicadores de situação do voo

Cria indicadores para identificar voos realizados e cancelados e valida a distribuição por situação.

In [0]:
fato_voos = (
    fato_voos
    .withColumn(
        "voo_realizado",
        F.col("situacao_voo") == "REALIZADO"
    )
    .withColumn(
        "voo_cancelado",
        F.col("situacao_voo") == "CANCELADO"
    )
)

display(
    fato_voos
    .groupBy("situacao_voo")
    .agg(
        F.count("*").alias("quantidade"),
        F.sum(F.col("voo_realizado").cast("int")).alias("realizados"),
        F.sum(F.col("voo_cancelado").cast("int")).alias("cancelados")
    )
    .orderBy("situacao_voo")
)

### Indicador de chegada pontual

Cria o indicador de chegada pontual, deixando o campo nulo para voos cancelados.

In [0]:
fato_voos = fato_voos.withColumn(
    "chegada_pontual",
    F.when(
        F.col("voo_cancelado"),
        None
    ).otherwise(
        F.col("atraso_chegada_min") <= 15
    )
)

### Validação da chegada pontual

Confere a distribuição do indicador de chegada pontual por situação do voo.

In [0]:
display(
    fato_voos.groupBy("situacao_voo", "chegada_pontual")
    .count()
    .orderBy("situacao_voo", "chegada_pontual")
)

### Validação de casos sem indicador

Procura voos realizados que ficaram sem classificação de chegada pontual para identificar possíveis problemas nos dados.

In [0]:
display(
    fato_voos
    .filter(
        (F.col("voo_realizado")) &
        (F.col("chegada_pontual").isNull())
    )
    .select(
        "situacao_voo",
        "chegada_prevista",
        "chegada_real",
        "atraso_chegada_min"
    )
    .limit(20)
)

### Mês de referência

Cria o mês de referência a partir da data prevista de partida, facilitando análises mensais.

In [0]:
fato_voos = fato_voos.withColumn(
    "mes_referencia",
    F.when(
        F.col("partida_prevista").isNotNull(),
        F.date_format(F.col("partida_prevista"), "yyyy-MM")
    )
)

### Validação do mês de referência

Confere alguns exemplos da data prevista e do mês de referência criado.

In [0]:
display(
    fato_voos.select(
        "partida_prevista",
        "mes_referencia"
    ).limit(20)
)

### Validação de datas nulas

Verifica os registros sem data prevista de partida e como o mês de referência ficou nesses casos.

In [0]:
display(
    fato_voos.filter(
        F.col("partida_prevista").isNull()
    ).select(
        "partida_prevista",
        "mes_referencia"
    ).limit(20)
)

### Conferência dos campos da fato

Exibe os principais campos criados e utilizados na fato para uma conferência geral.

In [0]:
display(
    fato_voos.select(
        "icao_empresa_aerea",
        "numero_voo",
        "partida_prevista",
        "partida_real",
        "chegada_prevista",
        "chegada_real",
        "atraso_partida_min",
        "atraso_chegada_min",
        "minutos_recuperados",
        "partida_pontual",
        "chegada_pontual",
        "voo_realizado",
        "voo_cancelado",
        "mes_referencia"
    ).limit(20)
)

### Atualização da fato de voos

Grava novamente a fato no Gold com as novas colunas e o schema atualizado.

In [0]:
fato_voos.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("voebem.gold.fato_voos")

### Conferência das tabelas Silver

Lista as tabelas disponíveis na Silver para identificar as dimensões que serão utilizadas na Gold.

In [0]:
spark.sql("""
    SHOW TABLES IN voebem.silver
""").show(truncate=False)

### Estrutura da dimensão de aeroportos

Consulta a estrutura do cadastro de aeroportos antes de montar a dimensão Gold.

In [0]:
display(spark.sql("""
    DESCRIBE TABLE voebem.silver.aerodromos
"""))

### Identificação dos aeroportos dos voos

Reúne os aeroportos de origem e destino presentes na fato, mantendo apenas os códigos distintos.

In [0]:
aeroportos_fato = spark.sql("""
    SELECT icao_aerodromo_origem AS icao
    FROM voebem.gold.fato_voos
    WHERE icao_aerodromo_origem IS NOT NULL

    UNION

    SELECT icao_aerodromo_destino AS icao
    FROM voebem.gold.fato_voos
    WHERE icao_aerodromo_destino IS NOT NULL
""")

print("Aeroportos distintos no fato:", aeroportos_fato.count())

### Validação do cadastro de aeroportos

Compara os aeroportos usados nos voos com o cadastro da Silver para identificar quais estão cadastrados.

In [0]:
display(
    aeroportos_fato.alias("f")
    .join(
        spark.table("voebem.silver.aerodromos").alias("a"),
        F.col("f.icao") == F.col("a.icao"),
        "left"
    )
    .select(
        F.col("f.icao"),
        F.col("a.nome"),
        F.when(
            F.col("a.icao").isNotNull(),
            "CADASTRADO"
        ).otherwise("NAO_CADASTRADO").alias("status")
    )
    .groupBy("status")
    .count()
)

### Validação de duplicidades

Verifica se existem códigos ICAO repetidos no cadastro de aeroportos da Silver.

In [0]:
display(
    spark.sql("""
        SELECT
            icao,
            COUNT(*) AS quantidade
        FROM voebem.silver.aerodromos
        WHERE icao IS NOT NULL
        GROUP BY icao
        HAVING COUNT(*) > 1
        ORDER BY quantidade DESC
    """)
)

### Criação da dimensão de aeroportos

Monta a dimensão de aeroportos usando os códigos encontrados na fato e enriquecendo os dados com o cadastro da Silver.

In [0]:
dim_aeroporto = (
    aeroportos_fato.alias("f")
    .join(
        spark.table("voebem.silver.aerodromos").alias("a"),
        F.col("f.icao") == F.col("a.icao"),
        "left"
    )
    .select(
        F.col("f.icao"),
        F.col("a.ciad"),
        F.col("a.nome"),
        F.col("a.municipio"),
        F.col("a.uf_nome"),
        F.col("a.municipio_servido"),
        F.col("a.uf_servido_nome"),
        F.col("a.latitude_dms"),
        F.col("a.longitude_dms"),
        F.col("a.altitude_m"),
        F.col("a.situacao")
    )
)

display(dim_aeroporto)

### Validação da dimensão de aeroportos

Confere a quantidade de aeroportos e separa os registros entre cadastrados e não cadastrados.

In [0]:
print("Total de aeroportos:", dim_aeroporto.count())

display(
    dim_aeroporto.groupBy(
        F.when(F.col("nome").isNotNull(), "CADASTRADO")
         .otherwise("NAO_CADASTRADO")
         .alias("status")
    ).count()
)

### Gravação da dimensão de aeroportos

Salva a dimensão de aeroportos no schema Gold.

In [0]:
dim_aeroporto.write \
    .mode("overwrite") \
    .saveAsTable("voebem.gold.dim_aeroporto")

### Conferência da dimensão de aeroportos

Exibe a dimensão gravada para validar o resultado final.

In [0]:
display(
    spark.table("voebem.gold.dim_aeroporto")
)

### Conferência da fato de voos

Exibe a fato de voos gravada no Gold antes da montagem da OBT.

In [0]:
display(
    spark.table("voebem.gold.fato_voos")
)

### Preparação das dimensões de aeroporto

Cria versões separadas da dimensão de aeroportos para permitir os joins de origem e destino.

In [0]:
dim_aeroporto_origem = (
    spark.table("voebem.gold.dim_aeroporto")
    .select(
        F.col("icao").alias("icao_origem"),
        F.col("nome").alias("nome_aeroporto_origem"),
        F.col("municipio").alias("municipio_origem"),
        F.col("uf_nome").alias("uf_origem")
    )
)

dim_aeroporto_destino = (
    spark.table("voebem.gold.dim_aeroporto")
    .select(
        F.col("icao").alias("icao_destino"),
        F.col("nome").alias("nome_aeroporto_destino"),
        F.col("municipio").alias("municipio_destino"),
        F.col("uf_nome").alias("uf_destino")
    )
)

### Dimensão de companhia

Seleciona o código ICAO e o nome das companhias a partir do cadastro de empresas da Silver.

In [0]:
# Dimensão de companhia
dim_empresa = (
    spark.table("voebem.silver.empresas")
    .select(
        F.col("icao").alias("icao_empresa_ref"),
        F.col("razao_social").alias("nome_companhia")
    )
)


### Dimensões de código DI e tipo de linha

Prepara os cadastros de código DI e tipo de linha para adicionar suas descrições aos voos.

In [0]:
# Dimensões de DI e tipo de linha
dim_di = (
    spark.table("voebem.silver.codigos_operacao")
    .filter(F.col("dominio") == "codigo_di")
    .select(
        F.col("codigo").alias("codigo_di_ref"),
        F.col("descricao").alias("descricao_di")
    )
)

dim_tipo_linha = (
    spark.table("voebem.silver.codigos_operacao")
    .filter(F.col("dominio") == "codigo_tipo_linha")
    .select(
        F.col("codigo").alias("codigo_tipo_linha_ref"),
        F.col("descricao").alias("descricao_tipo_linha")
    )
)


### Base da OBT

Define a fato de voos como base da OBT, mantendo uma linha para cada etapa de voo.

In [0]:
# Base da OBT: uma linha por etapa de voo
obt_voos = fato_voos


### Enriquecimento com aeroporto de origem

Adiciona à OBT o nome, município e UF do aeroporto de origem.

In [0]:
# Join da dimensão de aeroporto de origem
obt_voos = (
    obt_voos.alias("v")
    .join(
        dim_aeroporto_origem.alias("o"),
        F.col("v.icao_aerodromo_origem") == F.col("o.icao_origem"),
        "left"
    )
    .select(
        "v.*",
        F.col("o.nome_aeroporto_origem"),
        F.col("o.municipio_origem"),
        F.col("o.uf_origem")
    )
)


### Enriquecimento com aeroporto de destino

Adiciona à OBT o nome, município e UF do aeroporto de destino.

In [0]:
# Join da dimensão de aeroporto de destino
obt_voos = (
    obt_voos.alias("v")
    .join(
        dim_aeroporto_destino.alias("d"),
        F.col("v.icao_aerodromo_destino") == F.col("d.icao_destino"),
        "left"
    )
    .select(
        "v.*",
        F.col("d.nome_aeroporto_destino"),
        F.col("d.municipio_destino"),
        F.col("d.uf_destino")
    )
)


### Enriquecimento com companhia

Adiciona o nome da companhia aérea e cria um texto de fallback quando ela não está cadastrada.

In [0]:
# Join da dimensão de companhia
obt_voos = (
    obt_voos.alias("v")
    .join(
        dim_empresa.alias("e"),
        F.col("v.icao_empresa_aerea") == F.col("e.icao_empresa_ref"),
        "left"
    )
    .select(
        "v.*",
        F.col("e.nome_companhia")
    )
)

obt_voos = obt_voos.withColumn(
    "nome_companhia",
    F.when(
        F.col("nome_companhia").isNull(),
        F.concat(
            F.lit("COMPANHIA NAO CADASTRADA - "),
            F.col("icao_empresa_aerea")
        )
    ).otherwise(F.col("nome_companhia"))
)


### Enriquecimento com código DI

Adiciona a descrição do código DI e identifica códigos que não estão catalogados.

In [0]:
# Join da descrição do código DI
obt_voos = (
    obt_voos.alias("v")
    .join(
        dim_di.alias("di"),
        F.col("v.codigo_autorizacao_di") == F.col("di.codigo_di_ref"),
        "left"
    )
    .select(
        "v.*",
        F.col("di.descricao_di")
    )
)

obt_voos = obt_voos.withColumn(
    "descricao_di",
    F.when(
        F.col("descricao_di").isNull(),
        F.concat(
            F.lit("CODIGO DI NAO CATALOGADO - "),
            F.col("codigo_autorizacao_di")
        )
    ).otherwise(F.col("descricao_di"))
)


### Enriquecimento com tipo de linha

Adiciona a descrição do tipo de linha e classifica o voo como doméstico ou internacional.

In [0]:
# Join da descrição do tipo de linha
obt_voos = (
    obt_voos.alias("v")
    .join(
        dim_tipo_linha.alias("tl"),
        F.col("v.codigo_tipo_linha") == F.col("tl.codigo_tipo_linha_ref"),
        "left"
    )
    .select(
        "v.*",
        F.col("tl.descricao_tipo_linha")
    )
)

obt_voos = obt_voos.withColumn(
    "escopo_voo",
    F.when(
        F.col("codigo_tipo_linha").isin("N", "C"),
        "Domestico"
    ).when(
        F.col("codigo_tipo_linha").isin("I", "G"),
        "Internacional"
    )
)


### Fallbacks e país dos aeroportos

Preenche nomes de aeroportos não encontrados no cadastro e identifica o país de origem e destino a partir do código ICAO.

In [0]:
# Fallbacks e país dos aeroportos
obt_voos = (
    obt_voos
    .withColumn(
        "nome_aeroporto_origem",
        F.when(
            F.col("nome_aeroporto_origem").isNull(),
            F.concat(
                F.lit("AEROPORTO FORA DO CADASTRO ANAC - "),
                F.col("icao_aerodromo_origem")
            )
        ).otherwise(F.col("nome_aeroporto_origem"))
    )
    .withColumn(
        "nome_aeroporto_destino",
        F.when(
            F.col("nome_aeroporto_destino").isNull(),
            F.concat(
                F.lit("AEROPORTO FORA DO CADASTRO ANAC - "),
                F.col("icao_aerodromo_destino")
            )
        ).otherwise(F.col("nome_aeroporto_destino"))
    )
    .withColumn(
        "pais_origem",
        F.when(
            F.col("icao_aerodromo_origem").startswith("SB"),
            "Brasil"
        ).otherwise("Exterior")
    )
    .withColumn(
        "pais_destino",
        F.when(
            F.col("icao_aerodromo_destino").startswith("SB"),
            "Brasil"
        ).otherwise("Exterior")
    )
)


### Criação das rotas

Cria identificadores de rota usando os códigos ICAO e os municípios de origem e destino.

In [0]:
# Rotas
obt_voos = (
    obt_voos
    .withColumn(
        "rota_icao",
        F.concat(
            F.col("icao_aerodromo_origem"),
            F.lit(" - "),
            F.col("icao_aerodromo_destino")
        )
    )
    .withColumn(
        "rota_municipios",
        F.concat(
            F.coalesce(F.col("municipio_origem"), F.col("icao_aerodromo_origem")),
            F.lit(" - "),
            F.coalesce(F.col("municipio_destino"), F.col("icao_aerodromo_destino"))
        )
    )
)


### Campos de tempo

Cria campos derivados de horário, hora, dia da semana e mês de referência para facilitar análises temporais.

In [0]:
# Campos de tempo
obt_voos = (
    obt_voos
    .withColumn(
        "partida_prevista_hora",
        F.date_format(F.col("partida_prevista"), "HH:mm")
    )
    .withColumn(
        "hora_partida_prevista",
        F.hour(F.col("partida_prevista"))
    )
    .withColumn(
        "dia_semana",
        F.when(F.dayofweek("partida_prevista") == 1, "domingo")
         .when(F.dayofweek("partida_prevista") == 2, "segunda-feira")
         .when(F.dayofweek("partida_prevista") == 3, "terça-feira")
         .when(F.dayofweek("partida_prevista") == 4, "quarta-feira")
         .when(F.dayofweek("partida_prevista") == 5, "quinta-feira")
         .when(F.dayofweek("partida_prevista") == 6, "sexta-feira")
         .when(F.dayofweek("partida_prevista") == 7, "sábado")
    )
    .withColumn(
        "mes_referencia",
        F.trunc(F.col("partida_prevista"), "month")
    )
)


### Validação de atrasos fora da faixa

Calcula novamente os atrasos para identificar diferenças de horário fora de uma faixa considerada plausível.

In [0]:
# Indicador de atraso fora da faixa plausível
# Calculamos novamente a diferença entre horários para identificar casos
# abaixo de -120 min ou acima de 1440 min, mesmo que os atrasos tratados
# da fato_voos estejam nulos por regra de qualidade.
atraso_partida_calculado = (
    (F.col("partida_real").cast("long") - F.col("partida_prevista").cast("long")) / 60
)

atraso_chegada_calculado = (
    (F.col("chegada_real").cast("long") - F.col("chegada_prevista").cast("long")) / 60
)

obt_voos = obt_voos.withColumn(
    "atraso_fora_de_faixa",
    (
        (atraso_partida_calculado < -120) |
        (atraso_partida_calculado > 1440) |
        (atraso_chegada_calculado < -120) |
        (atraso_chegada_calculado > 1440)
    )
)


### Seleção final da OBT

Seleciona e organiza os campos finais que serão disponibilizados na tabela analítica de voos.

In [0]:
# Seleção final da OBT
obt_voos = obt_voos.select(
    F.col("icao_empresa_aerea").alias("icao_empresa"),
    F.col("numero_voo"),
    F.col("codigo_autorizacao_di").alias("codigo_di"),
    F.col("descricao_di"),
    F.col("codigo_tipo_linha"),
    F.col("descricao_tipo_linha"),
    F.col("escopo_voo"),
    F.col("nome_companhia"),
    F.col("icao_aerodromo_origem").alias("icao_origem"),
    F.col("nome_aeroporto_origem"),
    F.col("municipio_origem"),
    F.col("uf_origem"),
    F.col("pais_origem"),
    F.col("icao_aerodromo_destino").alias("icao_destino"),
    F.col("nome_aeroporto_destino"),
    F.col("municipio_destino"),
    F.col("uf_destino"),
    F.col("pais_destino"),
    F.col("rota_icao"),
    F.col("rota_municipios"),
    F.col("partida_prevista"),
    F.col("partida_prevista_data"),
    F.col("partida_prevista_hora"),
    F.col("hora_partida_prevista"),
    F.col("dia_semana"),
    F.col("mes_referencia"),
    F.col("partida_real"),
    F.col("chegada_prevista"),
    F.col("chegada_real"),
    F.col("atraso_partida_min"),
    F.col("atraso_chegada_min"),
    F.col("minutos_recuperados"),
    F.col("atraso_fora_de_faixa"),
    F.col("partida_pontual"),
    F.col("chegada_pontual"),
    F.col("situacao_voo"),
    F.col("voo_realizado"),
    F.col("voo_cancelado"),
    F.current_timestamp().alias("_processado_em")
)


### Validação da OBT

Compara a quantidade de registros com a fato e confere o schema final da OBT.

In [0]:
# Validação da OBT
print("Linhas fato_voos:", fato_voos.count())
print("Linhas obt_voos:", obt_voos.count())
obt_voos.printSchema()


### Persistência da OBT

Salva a OBT final no schema Gold como tabela pronta para consumo analítico.

In [0]:
# Persistência da OBT
obt_voos.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("voebem.gold.obt_voos")
